[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/danpele/SFM/blob/main/Quantlets/Ch_14/SFM_ch14_crypto_var/SFM_ch14_crypto_var.ipynb)

# SFM_ch14_crypto_var

VaR 1% and ES 2.5% (VaR_alpha = -q_alpha, the loss exceeded with probability alpha) of a position of 100 000 USD in Bitcoin, Ethereum and the S&P 500 by historical simulation, Normal, Student-t (maximum likelihood) and GARCH(1,1)-t for the next day; conversion of log-return VaR into USD; 10-day VaR against the square-root-of-time rule; a backtest of one-day Bitcoin VaR 1% since 2018 (historical simulation on 365 days and GARCH(1,1)-t re-estimated every 365 days) with the Kupiec test.

*Statistics of Financial Markets (SFM), Chapter 14: Crypto assets. Author: Daniel Traian Pele.*

> **Working in Google Colab? Save your own copy first.**
> The Colab link opens a temporary copy: when you close the tab or the session times out, your changes are lost.
> Use **File → Save a copy in Drive** (it goes to the *Colab Notebooks* folder of your ASE Google Drive) and work in that copy.
> For the team project, **File → Save a copy in GitHub** puts the notebook directly in your team's repository.
> Files you create while the notebook runs (CSV, charts) are also temporary: set `SAVE_TO_DRIVE = True` in the next cell to keep them.

In [ ]:
# Optional (Colab only): keep the files you create in Google Drive
SAVE_TO_DRIVE = False   # set to True, run this cell and allow access to your Drive

import os
OUT_DIR = '.'
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    OUT_DIR = '/content/drive/MyDrive/SFM/Chapter_14'
    os.makedirs(OUT_DIR, exist_ok=True)
print('Save your outputs to:', os.path.abspath(OUT_DIR))


In [ ]:
# the arch package (maximum-likelihood estimation of GARCH models) is not part of Google Colab
!pip install -q arch

In [ ]:
import os
import re
import json
import urllib.request
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.colors import to_rgb
from scipy import optimize
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

## Chart style

In [ ]:
# Palette (RGB values of latex/preamble.tex)
MainBlue = '#1A3A6E'
IDAred = '#CD0000'
Forest = '#2E7D32'
Amber = '#B5853F'
Orange = '#E67E22'
Purple = '#8E44AD'
Teal = '#17A2B8'
Crimson = '#DC3545'
DarkText = '#1F2A44'          # text, axes and ticks (dark navy, not grey)

PALETTE = [MainBlue, IDAred, Forest, Amber, Purple, Orange, Teal, Crimson]
# fixed colours for the series used in several chapters
COL = {'sp500': MainBlue, 'bet': IDAred, 'bettr': Orange, 'dax': Teal, 'btc': Amber, 'eth': Crimson,
       'eurron': Forest, 'gold': Purple, 'vix': Crimson, 'stoxx50': Forest, 'ndx': Teal}


def apply():
    """Set the course style for matplotlib."""
    rc = plt.rcParams
    rc['figure.facecolor'] = 'none'
    rc['axes.facecolor'] = 'none'
    rc['savefig.facecolor'] = 'none'
    rc['savefig.transparent'] = True
    rc['axes.grid'] = False
    rc['font.family'] = 'sans-serif'
    rc['font.sans-serif'] = ['Helvetica', 'Arial', 'DejaVu Sans']
    # font sizes for slides (charts are 9-11 inches wide and shown at 8-14 cm)
    rc['font.size'] = 12
    rc['axes.labelsize'] = 13
    rc['axes.titlesize'] = 13
    rc['xtick.labelsize'] = 11.5
    rc['ytick.labelsize'] = 11.5
    rc['legend.fontsize'] = 11
    rc['axes.spines.top'] = False
    rc['axes.spines.right'] = False
    rc['axes.linewidth'] = 0.6
    rc['lines.linewidth'] = 1.4
    rc['legend.facecolor'] = 'none'
    rc['legend.framealpha'] = 0
    rc['legend.frameon'] = False
    for k in ('text.color', 'axes.labelcolor', 'axes.edgecolor', 'xtick.color', 'ytick.color', 'axes.titlecolor'):
        rc[k] = DarkText
    rc['axes.prop_cycle'] = mpl.cycler(color=PALETTE)


def legend_outside_bottom(ax, ncol=2, y=-0.22, **kw):
    """Place the legend outside the plot, bottom centre (for a figure with several axes, pass the last one
    or use fig.legend with the same arguments)."""
    return ax.legend(loc='upper center', bbox_to_anchor=(0.5, y), ncol=ncol, frameon=False, **kw)


def fig_legend_bottom(fig, handles=None, labels=None, ncol=3, y=-0.02):
    """One legend for a whole figure (several panels), below the panels."""
    if handles is None:
        handles, labels = [], []
        for ax in fig.axes:
            for h, l in zip(*ax.get_legend_handles_labels()):
                if l not in labels and not l.startswith('_'):
                    handles.append(h)
                    labels.append(l)
    return fig.legend(handles, labels, loc='upper center', bbox_to_anchor=(0.5, y), ncol=ncol, frameon=False)


def _is_grey(c, tol=0.06):
    try:
        r, g, b = to_rgb(c)
    except ValueError:
        return False
    return max(r, g, b) - min(r, g, b) < tol and 0.25 < (r + g + b) / 3 < 0.95


def check_no_grey(fig):
    """House rule: no grey series and no grey text. Raises ValueError listing the offending elements."""
    bad = []
    for ax in fig.axes:
        for ln in ax.get_lines():
            if not ln.get_label().startswith('_') and _is_grey(ln.get_color()):
                bad.append(f'line {ln.get_label()!r}')
        for coll in ax.collections:
            fc = coll.get_facecolor()
            if len(fc) and coll.get_label() and not coll.get_label().startswith('_') and _is_grey(fc[0][:3]):
                bad.append(f'series {coll.get_label()!r}')
        for t in ax.texts + [ax.title, ax.xaxis.label, ax.yaxis.label]:
            if t.get_text() and _is_grey(t.get_color()):
                bad.append(f'text {t.get_text()[:30]!r}')
    if bad:
        raise ValueError('grey elements: ' + ', '.join(bad))


def save_fig(name, out_dir=None, show=True):
    """Save the figure as transparent PDF and PNG (OUT_DIR if defined, else the current folder), then show it."""
    d = out_dir or globals().get('OUT_DIR', '.')
    plt.savefig(os.path.join(d, f'{name}.pdf'), bbox_inches='tight', transparent=True)
    plt.savefig(os.path.join(d, f'{name}.png'), bbox_inches='tight', transparent=True, dpi=180)
    if show:
        plt.show()
    plt.close()


apply()
# the chapter scripts call the style as st.<name> (import sfm_style as st): the same names here
import types
st = types.SimpleNamespace(COL=COL, PALETTE=PALETTE, MainBlue=MainBlue, IDAred=IDAred, Forest=Forest, Amber=Amber,
                           Orange=Orange, Purple=Purple, Teal=Teal, Crimson=Crimson, DarkText=DarkText, apply=apply, legend_outside_bottom=legend_outside_bottom,
                           fig_legend_bottom=fig_legend_bottom, save_fig=save_fig, check_no_grey=check_no_grey)

## Data

In [ ]:
REPO_RAW = 'https://raw.githubusercontent.com/danpele/SFM/main/data/market/'
# local copy of the course data, if the notebook runs inside the repository
MARKET_DIR = next((os.path.join(d, 'data', 'market') for d in ('.', '..', '../..', '../../..')
                   if os.path.isdir(os.path.join(d, 'data', 'market'))), '')
END = '2026-09-18'          # last day of the saved data

# name -> (symbol, label, group, field, default start)
SERIES = {
    # equity indices
    'sp500':    ('GSPC.INDX',     'S&P 500',               'Equity', 'close', '2000-01-01'),
    'ndx':      ('NDX.INDX',      'Nasdaq 100',            'Equity', 'close', '2000-01-01'),
    'dax':      ('GDAXI.INDX',    'DAX',                   'Equity', 'close', '2000-01-01'),
    'stoxx50':  ('STOXX50E.INDX', 'Euro Stoxx 50',         'Equity', 'close', '2000-01-01'),
    'nikkei':   ('N225.INDX',     'Nikkei 225',            'Equity', 'close', '2000-01-01'),
    'bet':      ('BET',           'BET',                   'Equity', 'close', '2000-01-01'),
    'bettr':    ('BETTR.INDX',    'BET-TR',                'Equity', 'close', '2014-09-23'),
    'betxt':    ('BETXT.INDX',    'BET-XT',                'Equity', 'close', '2011-08-12'),
    'betfi':    ('BETFI.INDX',    'BET-FI',                'Equity', 'close', '2012-01-25'),
    'wig20':    ('WIG20.INDX',    'WIG20',                 'Equity', 'close', '2000-01-01'),
    'bux':      ('BUX.INDX',      'BUX',                   'Equity', 'close', '2000-01-01'),
    'px':       ('PX.INDX',       'PX',                    'Equity', 'close', '2000-01-01'),
    'vix':      ('VIX.INDX',      'VIX',                   'Volatility', 'close', '2000-01-01'),
    # ETFs (adjusted close)
    'spy':      ('SPY.US',        'SPY',                   'ETF', 'adjusted_close', '2000-01-01'),
    'tlt':      ('TLT.US',        'TLT (US Treasuries 20y+)', 'ETF', 'adjusted_close', '2002-07-30'),
    'gld':      ('GLD.US',        'GLD (gold ETF)',        'ETF', 'adjusted_close', '2004-11-18'),
    'tvbetetf': ('TVBETETF.RO',   'TVBETETF',              'ETF', 'adjusted_close', '2015-06-12'),
    # Bucharest Stock Exchange (adjusted close)
    'tlv':      ('TLV.RO',        'Banca Transilvania',    'Stock', 'adjusted_close', '2000-01-01'),
    'snp':      ('SNP.RO',        'OMV Petrom',            'Stock', 'adjusted_close', '2001-09-04'),
    'brd':      ('BRD.RO',        'BRD',                   'Stock', 'adjusted_close', '2001-01-16'),
    'tgn':      ('TGN.RO',        'Transgaz',              'Stock', 'adjusted_close', '2008-01-25'),
    'sng':      ('SNG.RO',        'Romgaz',                'Stock', 'adjusted_close', '2013-11-12'),
    'snn':      ('SNN.RO',        'Nuclearelectrica',      'Stock', 'adjusted_close', '2013-11-04'),
    'h2o':      ('H2O.RO',        'Hidroelectrica',        'Stock', 'adjusted_close', '2023-07-12'),
    # US stocks (adjusted close)
    'aapl':     ('AAPL.US',       'Apple',                 'Stock', 'adjusted_close', '2000-01-01'),
    'msft':     ('MSFT.US',       'Microsoft',             'Stock', 'adjusted_close', '2000-01-01'),
    'nvda':     ('NVDA.US',       'NVIDIA',                'Stock', 'adjusted_close', '2000-01-01'),
    'jpm':      ('JPM.US',        'JPMorgan Chase',        'Stock', 'adjusted_close', '2000-01-01'),
    # crypto (close, 7 days a week)
    'btc':      ('BTC-USD.CC',    'Bitcoin',               'Crypto', 'close', '2014-09-17'),
    'eth':      ('ETH-USD.CC',    'Ethereum',              'Crypto', 'close', '2015-08-07'),
    'sol':      ('SOL-USD.CC',    'Solana',                'Crypto', 'close', '2020-04-11'),
    'usdt':     ('USDT-USD.CC',   'Tether (USDT)',         'Crypto', 'close', '2015-02-26'),
    # FX and commodities
    'eurron':   ('REF:EUR',       'EUR/RON (BNR reference)', 'FX', 'close', '2005-07-01'),
    'usdron':   ('REF:USD',       'USD/RON (BNR reference)', 'FX', 'close', '2005-07-01'),
    'eurron_eodhd': ('EURRON.FOREX', 'EUR/RON (EODHD)',    'FX', 'close', '2005-07-01'),
    'eurusd':   ('EURUSD.FOREX',  'EUR/USD',               'FX', 'close', '2002-05-06'),
    'gold':     ('XAUUSD.FOREX',  'Gold (XAU/USD)',        'Commodity', 'close', '2000-01-01'),
    # government bond yields (close, % p.a.)
    'us10y':    ('US10Y.GBOND',   'US 10Y yield',          'Yield', 'close', '2000-01-01'),
    'de10y':    ('DE10Y.GBOND',   'Germany 10Y yield',     'Yield', 'close', '2007-03-28'),
    'ro10y':    ('RO10Y.GBOND',   'Romania 10Y yield',     'Yield', 'close', '2007-08-17'),
}
LABELS = {k: v[1] for k, v in SERIES.items()}
_CACHE = {}


def read_market(symbol):
    """Daily table of one symbol: local copy of the course data, otherwise the SFM repository on GitHub."""
    fname = f'{symbol}.csv'
    path = os.path.join(MARKET_DIR, fname) if MARKET_DIR else ''
    src = path if path and os.path.exists(path) else REPO_RAW + fname
    return pd.read_csv(src, index_col='date', parse_dates=True).sort_index()


def read_reference_rate(currency='EUR', start='2005-07-01', end=END):
    """Official BNR reference rate (RON per unit of currency), from the public yearly XML archives."""
    key = (currency, start, end)
    if key in _CACHE:
        return _CACHE[key]
    rows = []
    for y in range(max(int(start[:4]), 2005), int(end[:4]) + 1):
        url = f'https://curs.bnr.ro/files/xml/years/nbrfxrates{y}.xml'
        xml = urllib.request.urlopen(urllib.request.Request(url, headers={'User-Agent': 'Mozilla'}),
                                     timeout=60).read().decode()
        for d, body in re.findall(r'<Cube date="([\d-]+)">(.*?)</Cube>', xml, re.S):
            m = re.search(rf'<Rate currency="{currency}"(?: multiplier="\d+")?>([\d.]+)</Rate>', body)
            if m:
                rows.append((d, float(m.group(1))))
    s = pd.DataFrame(rows, columns=['date', 'close']).drop_duplicates('date').set_index('date')['close']
    s.index = pd.to_datetime(s.index)
    _CACHE[key] = s.sort_index().loc[start:end]
    return _CACHE[key]


def load_close(name, start=None, end=END, field=None):
    """Daily closing price of a named series, cleaned with the course conventions."""
    symbol, label, group, field0, start0 = SERIES[name]
    start, field = start or start0, field or field0
    if symbol.startswith('REF:'):
        return read_reference_rate(symbol.split(':')[1], start=start, end=end).rename(name)
    t = read_market(symbol)
    s = (t[field] if field in t.columns else t['close']).loc[start:end]
    s = pd.to_numeric(s, errors='coerce').dropna()
    if group != 'Yield':
        s = s[s > 0]
    if group != 'Crypto':
        s = s[s.index.dayofweek < 5]          # no weekend quotes
        s = s[s.diff() != 0]                  # no holidays filled with the previous price
    s.index.name = 'date'
    return s.rename(name)


def load_ohlc(name, start=None, end=END):
    """Open, high, low, close (and volume) of a series with OHLC data (range-based volatility estimators)."""
    symbol, _, group, _, start0 = SERIES[name]
    t = read_market(symbol).loc[start or start0:end]
    cols = [c for c in ('open', 'high', 'low', 'close', 'volume') if c in t.columns]
    t = t[cols].dropna()
    if group != 'Crypto':
        t = t[t.index.dayofweek < 5]
    return t[(t[['open', 'high', 'low', 'close']] > 0).all(axis=1)] if 'open' in cols else t


def log_returns(name, start=None, end=END, pct=True):
    """Daily log returns r_t = ln P_t - ln P_{t-1} (in % by default), on the series' own calendar."""
    r = np.log(load_close(name, start, end)).diff().dropna()
    return (100 * r if pct else r).rename(name)


def simple_returns(name, start=None, end=END, pct=True):
    """Daily simple returns R_t = P_t / P_{t-1} - 1 (in % by default)."""
    r = load_close(name, start, end).pct_change().dropna()
    return (100 * r if pct else r).rename(name)


def load_panel(names, start=None, end=END, kind='close'):
    """Several series aligned on date: kind = 'close' or 'returns' (NaN where a market does not trade)."""
    f = load_close if kind == 'close' else log_returns
    return pd.concat([f(n, start, end) for n in names], axis=1)


def periods_per_year(r):
    """Average number of observations per calendar year (the actual frequency of the series)."""
    years = (r.index[-1] - r.index[0]).days / 365.25
    return len(r) / years

## Analysis

In [ ]:
import matplotlib.dates as mdates

import statsmodels.api as sm

from arch import arch_model

EXTRA = {'usdc': ('USDC-USD.CC', 'USD Coin (USDC)', 'Crypto', 'close', '2018-10-09'), 'dai': ('DAI-USD.CC', 'Dai (DAI)', 'Crypto', 'close', '2017-12-27'), 'ibit': ('IBIT.US', 'iShares Bitcoin Trust (IBIT)', 'ETF', 'adjusted_close', '2024-01-11')}

NAME = {'btc': 'Bitcoin', 'eth': 'Ethereum', 'sol': 'Solana', 'sp500': 'S&P 500', 'gold': 'Gold', 'usdt': 'Tether (USDT)', 'usdc': 'USD Coin (USDC)', 'dai': 'Dai (DAI)', 'ibit': 'IBIT'}

START = {'btc': '2014-09-17', 'eth': '2016-01-01', 'sol': '2020-04-11', 'sp500': '2014-09-17', 'gold': '2014-09-17'}

ASSETS = ['btc', 'eth', 'sol', 'sp500', 'gold']

COLORS = {'btc': '#E67E22', 'eth': '#8E44AD', 'sol': '#17A2B8', 'sp500': '#1A3A6E', 'gold': '#B5853F', 'usdt': '#2E7D32', 'usdc': '#1A3A6E', 'dai': '#E67E22', 'total': '#CD0000', 'ibit': '#1A3A6E'}

HALVINGS = ['2012-11-28', '2016-07-09', '2020-05-11', '2024-04-20']

ETF_DATE = '2024-01-11'

EVENTS = {'COVID-19': '2020-03-12', 'Terra/Luna': '2022-05-09', 'FTX': '2022-11-08', 'spot ETFs': '2024-01-11'}

EPISODES = {'COVID-19': ('2020-02-19', '2020-03-23'), 'Terra/Luna': ('2022-05-05', '2022-05-12'), 'FTX': ('2022-11-05', '2022-11-21')}

HILL_FRAC = 0.025

VR_Q = (2, 5, 10)

VR_WINDOW, VR_STEP = 500, 21

SUBPERIODS = [('2014-09-17', '2017-12-31'), ('2018-01-01', '2021-12-31'), ('2022-01-01', '2026-09-18')]

CORR_WINDOW = 250

ALPHA_VAR, ALPHA_ES = 0.01, 0.025

POSITION = 100000

BT_START = '2018-01-01'

BT_WINDOW = 365

PEG_START = '2021-01-01'

SVB = ('2023-03-08', '2023-03-20')

LLAMA = 'https://stablecoins.llama.fi/stablecoincharts/all'

STABLE_ID = {'usdt': 1, 'usdc': 2, 'ust': 3, 'dai': 5}

DAYS = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']

WD_PERIODS = {'2014-2019': ('2014-09-17', '2019-12-31'), '2020-2023': ('2020-01-01', '2023-12-31'), '2024-2026': ('2024-01-01', '2026-09-18')}

SEED = 2026

_LLAMA = {}

TABLE_DIR = '.'

def register_extra():
    """Add USDC, DAI and the IBIT ETF to the list of named series of the course data loader."""
    for k, v in EXTRA.items():
        SERIES.setdefault(k, v)


def close(k, start=None, end=END):
    """Daily close (adjusted close for the ETF) on the series' own calendar (crypto: 7 days a week)."""
    register_extra()
    return load_close(k, start or START.get(k), end)


def returns(k, start=None, end=END):
    """Daily log returns in % on the series' own calendar."""
    register_extra()
    return log_returns(k, start or START.get(k), end)


def joint_returns(a, b, start=None, end=END):
    """Log returns in % of two series on their common days: prices are joined first, then differenced."""
    p = pd.concat([close(a, start, end), close(b, start, end)], axis=1).dropna()
    return (100 * np.log(p).diff()).dropna()


def garch_fit(r, o=0):
    """GARCH(1,1) (o=0) or GJR-GARCH(1,1) (o=1) with constant mean and standardised Student-t innovations."""
    am = arch_model(r, mean='Constant', vol='GARCH', p=1, o=o, q=1, dist='t')
    return am.fit(disp='off', options={'maxiter': 2000})


def hs_var(x, a=ALPHA_VAR):
    """Historical-simulation VaR_a = -r_(k), k = ceil(n a) (a positive loss, in %)."""
    s = np.sort(np.asarray(x, float))
    return float(-s[int(np.ceil(len(s) * a)) - 1])


def hs_es(x, a=ALPHA_ES):
    """Historical-simulation ES_a = minus the mean of the k = ceil(n a) smallest returns."""
    s = np.sort(np.asarray(x, float))
    return float(-s[:int(np.ceil(len(s) * a))].mean())


def std_t_q(nu, a):
    """a-quantile of the Student-t scaled to unit variance."""
    return float(stats.t.ppf(a, nu) * np.sqrt((nu - 2) / nu))


def std_t_es(nu, a):
    """ES_a of the unit-variance Student-t (a positive number)."""
    q = stats.t.ppf(a, nu)
    return float(np.sqrt((nu - 2) / nu) * stats.t.pdf(q, nu) / a * (nu + q ** 2) / (nu - 1))


def var_table(names=('btc', 'eth', 'sp500'), W=POSITION):
    """VaR 1% and ES 2.5% (in % and in USD for a position of W) by historical simulation, Normal, Student-t (maximum
    likelihood) on the whole sample and GARCH(1,1)-t for the next day; historical 10-day VaR 1% on overlapping
    10-day returns against the square-root-of-time rule."""
    out = {}
    for k in names:
        r = returns(k)
        x = r.values
        mu, sd = x.mean(), x.std(ddof=1)
        nu, loc, sc = stats.t.fit(x)
        res = garch_fit(r)
        p = res.params
        s1 = float(np.sqrt(res.forecast(horizon=1, reindex=False).variance.values[-1, 0]))
        r10 = np.convolve(x, np.ones(10), mode='valid')
        d = {'n': int(len(x)), 'hs_v': hs_var(x), 'hs_e': hs_es(x),
             'n_v': float(-(mu + sd * stats.norm.ppf(ALPHA_VAR))),
             'n_e': float(-mu + sd * stats.norm.pdf(stats.norm.ppf(ALPHA_ES)) / ALPHA_ES),
             't_v': float(-(loc + sc * stats.t.ppf(ALPHA_VAR, nu))),
             't_e': float(-loc + sc * stats.t.pdf(stats.t.ppf(ALPHA_ES, nu), nu) / ALPHA_ES * (nu + stats.t.ppf(ALPHA_ES, nu) ** 2) / (nu - 1)),
             'nu': float(nu), 'g_sigma': s1, 'g_nu': float(p['nu']),
             'g_v': float(-(p['mu'] + s1 * std_t_q(p['nu'], ALPHA_VAR))), 'g_e': float(-p['mu'] + s1 * std_t_es(p['nu'], ALPHA_ES)),
             'hs10': hs_var(r10), 'sqrt10': float(np.sqrt(10) * hs_var(x)), 'last': r.index[-1].date().isoformat(),
             'mu': float(mu), 'sd': float(sd)}
        for c in ['hs_v', 'hs_e', 'n_v', 'n_e', 't_v', 't_e', 'g_v', 'g_e', 'hs10']:
            d[c + '_usd'] = float(W * (1 - np.exp(-d[c] / 100)))
        out[k] = d
    return out


def kupiec(x, n, a=ALPHA_VAR):
    """Kupiec (1995) proportion-of-failures LR test: x exceptions in n days."""
    pi = x / n
    l0 = (n - x) * np.log(1 - a) + x * np.log(a)
    l1 = (n - x) * np.log(1 - pi) + (x * np.log(pi) if x > 0 else 0.0)
    lr = -2 * (l0 - l1)
    return float(lr), float(stats.chi2.sf(lr, 1))


def backtest_btc(start=BT_START, window=BT_WINDOW, k='btc'):
    """One-day VaR 1% forecasts for Bitcoin since `start`: historical simulation on the last 365 days and
    GARCH(1,1)-t re-estimated every 365 days on all earlier data; exceptions, rate, Kupiec test, exceptions by year
    (another crypto asset with k='eth')."""
    r = returns(k)
    idx = np.where(r.index >= pd.Timestamp(start))[0]
    hs = pd.Series([hs_var(r.values[i - window:i]) for i in idx], index=r.index[idx])
    g = pd.Series(index=r.index[idx], dtype=float)
    for j0 in range(0, len(idx), window):
        i0 = idx[j0]
        am = arch_model(r, mean='Constant', vol='GARCH', p=1, q=1, dist='t')
        res = am.fit(disp='off', last_obs=r.index[i0], options={'maxiter': 2000})
        f = res.forecast(horizon=1, start=r.index[i0 - 1], reindex=False)
        p = res.params
        q = std_t_q(p['nu'], ALPHA_VAR)
        sig = np.sqrt(f.variance.values[:, 0])
        # forecast made on day i-1 for day i
        fc = pd.Series(-(p['mu'] + sig * q), index=r.index[i0 - 1:])[:-1]
        fc.index = r.index[i0:]
        sl = r.index[idx[j0:j0 + window]]
        g.loc[sl] = fc.loc[sl].values
    out = {'start': r.index[idx[0]].date().isoformat(), 'n': int(len(idx))}
    df = pd.DataFrame({'r': r.iloc[idx], 'HS': hs, 'GARCH-t': g})
    for m in ('HS', 'GARCH-t'):
        hit = df['r'] < -df[m]
        x = int(hit.sum())
        lr, pv = kupiec(x, len(df))
        out[m] = {'x': x, 'rate': float(100 * x / len(df)), 'exp': float(0.01 * len(df)), 'lr': lr, 'p': pv,
                  'year': {str(y): int(v) for y, v in hit.groupby(df.index.year).sum().items()},
                  'mean_var': float(df[m].mean())}
    return out, df


def fig_btc_var(df, save=True):
    """Bitcoin daily returns since 2018 with minus VaR 1% from historical simulation (365 days) and GARCH(1,1)-t."""
    fig, ax = plt.subplots(figsize=(11, 3.9))
    ax.plot(df.index, df['r'], color=st.MainBlue, lw=0.5, alpha=0.8, label='Bitcoin daily return (%)')
    ax.plot(df.index, -df['HS'], color=st.Amber, lw=1.3, label='minus VaR 1%: historical simulation, 365 days')
    ax.plot(df.index, -df['GARCH-t'], color=st.IDAred, lw=1.0, label='minus VaR 1%: GARCH(1,1)-t')
    hit = df['r'] < -df['HS']
    ax.scatter(df.index[hit], df['r'][hit], s=14, color=st.Orange, zorder=3, label='Exception of the historical VaR')
    ax.set_ylim(-45, 25)
    ax.set_ylabel('%')
    st.legend_outside_bottom(ax, ncol=2, y=-0.12)
    st.check_no_grey(fig)
    if save:
        st.save_fig('sfm_ch14_btc_var')

## Run

In [ ]:
V = var_table()
print(pd.DataFrame(V).T[['hs_v', 'hs_e', 'n_v', 't_v', 'g_v', 'hs_v_usd', 'hs10', 'sqrt10']].round(2))
bt, df = backtest_btc()
print(bt)
fig_btc_var(df)

![sfm_ch14_btc_var](./sfm_ch14_btc_var.png)

Output: `sfm_ch14_btc_var.pdf`